# EDA 003.03 — Creating Features

**Kaggle Feature Engineering Course — Lesson 3**
**Reference:** [Creating Features](https://www.kaggle.com/code/ryanholbrook/creating-features) by Ryan Holbrook

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Apply **mathematical transforms** (log, sqrt, power, reciprocal)
2. Create **interaction features** (products, ratios, differences)
3. Build **aggregation features** using groupby statistics
4. Engineer features from **counts and flags**
5. Understand **when** each technique is most appropriate

---

## Overview of Techniques

```
Creating Features
├── Mathematical Transforms
│   ├── log, sqrt, power        → handle skewed distributions
│   └── reciprocal, abs         → domain-specific
├── Interaction Features
│   ├── X1 × X2                 → combined effect
│   ├── X1 / X2                 → ratio (normalises by another feature)
│   └── X1 - X2                 → difference / change
├── Aggregation Features
│   └── groupby(cat).agg(mean/std/count)  → encode group statistics
└── Count / Flag Features
    ├── count of non-null columns
    └── binary flags (is_zero, is_weekend, has_X)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## 1 — Mathematical Transforms

### When to use
- **`log(x)`** — right-skewed features (income, population, house price); compresses large values, expands small ones
- **`sqrt(x)`** — counts and frequencies; milder compression than log
- **`x²` / `x³`** — when you expect curvature in the relationship
- **`1/x`** — rate features (e.g. time-per-unit → units-per-time)

> **Rule of thumb:** If `skewness > 1`, try a log transform. Check that the feature is strictly positive first (`log` is undefined for 0 and negative values — use `log1p` which computes `log(x + 1)`).

In [ ]:
np.random.seed(1)
# Simulate highly skewed salary data
salary = np.random.exponential(scale=30000, size=500) + 20000

df = pd.DataFrame({'salary': salary})
df['log_salary']  = np.log(df['salary'])
df['sqrt_salary'] = np.sqrt(df['salary'])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
configs = [
    ('salary',      'steelblue',  f'Raw Salary  (skew={salary.mean()/salary.std():.1f})'),
    ('log_salary',  'darkorange', 'log(salary)'),
    ('sqrt_salary', 'seagreen',   'sqrt(salary)'),
]
for ax, (col, color, title) in zip(axes, configs):
    ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    skew = df[col].skew()
    ax.set_title(f'{title}\nskewness = {skew:.2f}', fontsize=11)

plt.tight_layout()
plt.show()
print("Original skewness :", round(df['salary'].skew(), 3))
print("After log         :", round(df['log_salary'].skew(), 3))
print("After sqrt        :", round(df['sqrt_salary'].skew(), 3))

---
## 2 — Interaction Features

Two features individually may be weak predictors, but their **combination** can be powerful.

### Examples
- `length × width = area` — captures the actual quantity of interest
- `price / area = price_per_sqm` — normalised, comparable across sizes
- `current_value - purchase_price = profit` — the meaningful difference
- `calls_made / days_active = call_rate` — rate rather than absolute count

In [ ]:
np.random.seed(42)
n = 400
length = np.random.uniform(5, 30, n)
width  = np.random.uniform(3, 20, n)
noise  = np.random.normal(0, 5, n)
# True target: price driven by AREA, not individual length/width
price  = 50 * length * width + noise

df = pd.DataFrame({'length': length, 'width': width, 'price': price})
df['area'] = df['length'] * df['width']   # interaction feature

results = {}
for features in [['length'], ['width'], ['length', 'width'], ['area']]:
    model = LinearRegression().fit(df[features], df['price'])
    r2 = r2_score(df['price'], model.predict(df[features]))
    results[str(features)] = r2

print("R² scores:")
for k, v in results.items():
    print(f"  {k:30s} → {v:.4f}")

**Observe:** The interaction feature `area = length × width` achieves near-perfect R², while individual features or their combination without the interaction still struggle.

---
## 3 — Aggregation Features (Group Statistics)

When data has a **group structure** (customers, cities, product categories), aggregating statistics per group can encode powerful contextual information.

| Aggregation | Meaning |
|---|---|
| `mean` | Typical value for the group |
| `std` | Variability within the group |
| `count` | Group size / popularity |
| `min/max` | Extremes within the group |

> **Important:** Compute aggregations on the **training set only**, then merge onto test — avoids data leakage.

In [ ]:
np.random.seed(7)
cities = ['New York', 'Chicago', 'Austin', 'Seattle', 'Miami']
city_base_price = {'New York': 800, 'Chicago': 450, 'Austin': 380, 'Seattle': 520, 'Miami': 600}

n = 500
city = np.random.choice(cities, n)
area = np.random.uniform(40, 200, n)
base = np.array([city_base_price[c] for c in city])
rent = base * (area / 80) + np.random.normal(0, 60, n)

df = pd.DataFrame({'city': city, 'area': area, 'rent': rent})

# --- Aggregation features: city-level statistics ---
city_stats = (df.groupby('city')['rent']
                .agg(city_mean_rent='mean', city_std_rent='std', city_count='count')
                .reset_index())

df = df.merge(city_stats, on='city')

# Derived: how does this listing compare to its city average?
df['rent_vs_city_mean'] = df['rent'] - df['city_mean_rent']
df['rent_ratio']        = df['rent'] / df['city_mean_rent']

print(city_stats.round(1))
print("\nSample with aggregation features:")
df[['city', 'area', 'rent', 'city_mean_rent', 'rent_vs_city_mean', 'rent_ratio']].head()

---
## 4 — Count and Flag Features

Simple but effective features for structured data:

- **Count of non-null** values in a row → proxy for 'completeness' of a customer profile
- **Binary flags** — `is_zero`, `is_weekend`, `has_email`, `is_first_purchase`
- **Count of specific values** — `num_previous_orders`, `num_failed_payments`

In [ ]:
df_customers = pd.DataFrame({
    'age':         [25, np.nan, 34, 45, np.nan],
    'income':      [50000, 70000, np.nan, 120000, 45000],
    'email':       ['a@b.com', np.nan, 'c@d.com', 'e@f.com', np.nan],
    'phone':       ['555-1234', '555-5678', '555-9101', np.nan, '555-1122'],
    'num_orders':  [3, 0, 7, 15, 1],
    'total_spend': [350.0, 0.0, 820.0, 4500.0, 90.0],
})

# Count of non-null columns (profile completeness)
df_customers['profile_completeness'] = df_customers.notna().sum(axis=1)

# Binary flags
df_customers['has_email']       = df_customers['email'].notna().astype(int)
df_customers['has_phone']       = df_customers['phone'].notna().astype(int)
df_customers['is_new_customer'] = (df_customers['num_orders'] <= 1).astype(int)

# Spend per order (avoid zero-division)
df_customers['avg_order_value'] = (df_customers['total_spend'] /
                                   df_customers['num_orders'].replace(0, np.nan))

df_customers

---
## Key Takeaways

- **Transform before modelling** — reduce skew in numeric features for linear models
- **Interaction features** capture multiplicative and ratio effects that individual features miss
- **Aggregation features** encode group context — a listing's price is partly determined by the city it's in
- **Count/flag features** are cheap to create and often surprisingly useful
- Always **validate** that a new feature improves CV score before adding it permanently

---

## Further Reading

- [Kaggle Tutorial — Creating Features](https://www.kaggle.com/code/ryanholbrook/creating-features)
- [Feature Engineering for Machine Learning](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/) — Zheng & Casari, Ch. 2–4
- [Feature Engineering and Selection](http://www.feat.engineering/creating-features.html) — Kuhn & Johnson
- [Sklearn — PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)